# Route Traffic Signal Map

This notebook visualises a BEB route on a real basemap and labels each stop-to-stop segment with the number of traffic signals from `traffic_signals.csv`.

It uses:

- a static GTFS zip for stop coordinates, trip stop order, and route shapes;
- the route result CSV produced by `gtfs_to_segment.py`, mainly to identify the processed trip IDs;
- the traffic signal cache CSV, keyed by `from_stop_id` and `to_stop_id`.

The map is saved as an interactive `.html` file.

In [17]:
from pathlib import Path
import csv
import html
import io
import math
import zipfile

import pandas as pd
import folium
import branca.colormap as cm

## 1. Set Input Paths

Update these paths if your files are stored elsewhere.

In [18]:
PROJECT_ROOT = Path.cwd()

# Static GTFS feed used by the model.
GTFS_ZIP = PROJECT_ROOT / "../data/raw/GTFS_Realtime.zip"

# Route result CSV generated by gtfs_to_segment.py, e.g.
# data/processed/route_208_trips_2025_07_12.csv
ROUTE_RESULTS_CSV = PROJECT_ROOT / "../data/processed/route_208_trips_2025_07_12.csv"

# Traffic signal cache CSV generated by traffic_signals.py.
TRAFFIC_SIGNALS_CSV = PROJECT_ROOT / "../data/processed/traffic_signals.csv"

# Plot one representative trip, or all unique stop-pair segments used in the CSV.
# Options: "first_trip", "busiest_trip", "all_unique_pairs"
PLOT_MODE = "first_trip"

OUTPUT_HTML = PROJECT_ROOT / "../reports/figures/route_traffic_signals_map.html"
OUTPUT_HTML.parent.mkdir(parents=True, exist_ok=True)

for name, path in {
    "GTFS_ZIP": GTFS_ZIP,
    "ROUTE_RESULTS_CSV": ROUTE_RESULTS_CSV,
    "TRAFFIC_SIGNALS_CSV": TRAFFIC_SIGNALS_CSV,
}.items():
    print(f"{name}: {path}  exists={path.exists()}")

GTFS_ZIP: c:\Users\ninglin.ou\beb\notebooks\..\data\raw\GTFS_Realtime.zip  exists=True
ROUTE_RESULTS_CSV: c:\Users\ninglin.ou\beb\notebooks\..\data\processed\route_208_trips_2025_07_12.csv  exists=True
TRAFFIC_SIGNALS_CSV: c:\Users\ninglin.ou\beb\notebooks\..\data\processed\traffic_signals.csv  exists=True


## 2. Helper Functions

In [19]:
EARTH_R = 6_371_000.0

def haversine_m(lat1, lon1, lat2, lon2):
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    return 2 * EARTH_R * math.asin(math.sqrt(a))

def gtfs_time_to_seconds(t):
    h, m, s = (int(x) for x in str(t).strip().split(":"))
    return h * 3600 + m * 60 + s

def read_gtfs_table(gtfs_zip, member, dtype=None):
    with zipfile.ZipFile(gtfs_zip) as z:
        with z.open(member) as f:
            return pd.read_csv(f, dtype=dtype)

def stop_times_for_trips(gtfs_zip, trip_ids):
    wanted = set(map(str, trip_ids))
    by_trip = {tid: [] for tid in wanted}
    with zipfile.ZipFile(gtfs_zip) as z:
        with z.open("stop_times.txt") as raw:
            reader = csv.DictReader(io.TextIOWrapper(raw, encoding="utf-8"))
            for row in reader:
                tid = row["trip_id"]
                if tid in by_trip:
                    by_trip[tid].append(row)
    for tid in by_trip:
        by_trip[tid].sort(key=lambda r: int(r["stop_sequence"]))
    return by_trip

def shape_points_with_cumdist(shape_rows):
    pts = []
    last = None
    cum_m = 0.0
    for row in shape_rows.sort_values("shape_pt_sequence").itertuples(index=False):
        lat = float(row.shape_pt_lat)
        lon = float(row.shape_pt_lon)
        if last is not None:
            cum_m += haversine_m(last[0], last[1], lat, lon)
        pts.append((lat, lon, cum_m))
        last = (lat, lon)
    return pts

def project_stop_to_shape_m(lat, lon, shape_points, min_cum_m=None):
    if not shape_points or len(shape_points) < 2:
        return None
    lat0 = math.radians(lat)
    best = None
    for a, b in zip(shape_points[:-1], shape_points[1:]):
        lat_a, lon_a, cum_a = a
        lat_b, lon_b, cum_b = b
        seg_len = cum_b - cum_a
        if seg_len <= 0:
            continue
        ax = math.radians(lon_a - lon) * math.cos(lat0) * EARTH_R
        ay = math.radians(lat_a - lat) * EARTH_R
        bx = math.radians(lon_b - lon) * math.cos(lat0) * EARTH_R
        by = math.radians(lat_b - lat) * EARTH_R
        vx, vy = bx - ax, by - ay
        denom = vx * vx + vy * vy
        if denom <= 0:
            continue
        t = max(0.0, min(1.0, -(ax * vx + ay * vy) / denom))
        cum = cum_a + t * seg_len
        if min_cum_m is not None and cum < min_cum_m:
            continue
        px, py = ax + t * vx, ay + t * vy
        dist2 = px * px + py * py
        if best is None or dist2 < best[0]:
            best = (dist2, cum)
    if best is None and min_cum_m is not None:
        return project_stop_to_shape_m(lat, lon, shape_points, min_cum_m=None)
    return None if best is None else best[1]

def interp_on_shape(shape_points, cum):
    if cum <= shape_points[0][2]:
        return (shape_points[0][0], shape_points[0][1])
    if cum >= shape_points[-1][2]:
        return (shape_points[-1][0], shape_points[-1][1])
    for (lat0, lon0, c0), (lat1, lon1, c1) in zip(shape_points[:-1], shape_points[1:]):
        if c0 <= cum <= c1 and c1 > c0:
            f = (cum - c0) / (c1 - c0)
            return (lat0 + f * (lat1 - lat0), lon0 + f * (lon1 - lon0))
    return (shape_points[-1][0], shape_points[-1][1])

def shape_subpath(shape_points, cum_a, cum_b):
    if not shape_points or cum_b <= cum_a:
        return None
    pts = [interp_on_shape(shape_points, cum_a)]
    for lat, lon, cum in shape_points:
        if cum_a < cum < cum_b:
            pts.append((lat, lon))
    pts.append(interp_on_shape(shape_points, cum_b))
    out = [pts[0]]
    for p in pts[1:]:
        if p != out[-1]:
            out.append(p)
    return out if len(out) >= 2 else None

def midpoint(polyline):
    return polyline[len(polyline) // 2]

## 3. Load Data

In [20]:
missing = [p for p in [GTFS_ZIP, ROUTE_RESULTS_CSV, TRAFFIC_SIGNALS_CSV] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Please update the input paths. Missing files:\n"
        + "\n".join(str(p) for p in missing)
    )

route_df = pd.read_csv(ROUTE_RESULTS_CSV, dtype={"trip_id": str, "from_stop_id": str, "to_stop_id": str})
sig_df = pd.read_csv(TRAFFIC_SIGNALS_CSV, dtype={"from_stop_id": str, "to_stop_id": str})
sig_df["n_signals"] = pd.to_numeric(sig_df["n_signals"], errors="coerce").fillna(0).astype(int)

stops = read_gtfs_table(GTFS_ZIP, "stops.txt", dtype={"stop_id": str})
trips = read_gtfs_table(GTFS_ZIP, "trips.txt", dtype={"trip_id": str, "route_id": str, "shape_id": str})
routes = read_gtfs_table(GTFS_ZIP, "routes.txt", dtype={"route_id": str})

with zipfile.ZipFile(GTFS_ZIP) as z:
    has_shapes = "shapes.txt" in set(z.namelist())
shapes = read_gtfs_table(GTFS_ZIP, "shapes.txt", dtype={"shape_id": str}) if has_shapes else None

print(f"Route result rows: {len(route_df):,}")
print(f"Traffic signal cache rows: {len(sig_df):,}")
print(f"GTFS stops: {len(stops):,}; trips: {len(trips):,}; shapes available: {has_shapes}")
route_df.head()

Route result rows: 4,727
Traffic signal cache rows: 83
GTFS stops: 10,213; trips: 246,672; shapes available: True


,route_short_name,route_long_name,route_id,service_id,block_id,duty_id,duty_index,duty_trip_index,duty_trip_count,duty_start_time,...,net_battery_energy_kWh,gross_consumed_kWh,regen_recovered_kWh,aux_energy_kWh,net_battery_kWh_per_km,gross_consumed_kWh_per_km,cum_dist_km,SoC_start_%,SoC_end_%,trip_end_soc_%
0,208,Lotabeg - Bishopstown - Curraheen,2 208 c a,246,5695_7778208_42080201,246:5695_7778208_42080201,0,0,19,07:45:00,...,0.832,0.862,0.055,0.025,3.788,3.925,0.220,100.00,99.80,95.52
1,208,Lotabeg - Bishopstown - Curraheen,2 208 c a,246,5695_7778208_42080201,246:5695_7778208_42080201,0,0,19,07:45:00,...,0.973,1.146,0.198,0.025,3.630,4.275,0.488,99.80,99.56,95.52
2,208,Lotabeg - Bishopstown - Curraheen,2 208 c a,246,5695_7778208_42080201,246:5695_7778208_42080201,0,0,19,07:45:00,...,0.184,0.503,0.343,0.025,0.731,1.995,0.740,99.56,99.51,95.52
3,208,Lotabeg - Bishopstown - Curraheen,2 208 c a,246,5695_7778208_42080201,246:5695_7778208_42080201,0,0,19,07:45:00,...,-0.109,0.045,0.179,0.025,-0.738,0.303,0.887,99.51,99.54,95.52
4,208,Lotabeg - Bishopstown - Curraheen,2 208 c a,246,5695_7778208_42080201,246:5695_7778208_42080201,0,0,19,07:45:00,...,0.355,0.338,0.033,0.050,1.067,1.015,1.220,99.54,99.45,95.52


## 4. Choose Trip or Route Segments

In [21]:
if "trip_id" not in route_df.columns:
    raise ValueError("ROUTE_RESULTS_CSV must contain a trip_id column.")

if PLOT_MODE == "first_trip":
    trip_ids = [str(route_df["trip_id"].dropna().iloc[0])]
elif PLOT_MODE == "busiest_trip":
    energy_col = "net_battery_energy_kWh" if "net_battery_energy_kWh" in route_df.columns else "energy_kWh"
    trip_ids = [
        str(
            route_df.groupby("trip_id")[energy_col]
            .sum()
            .sort_values(ascending=False)
            .index[0]
        )
    ]
elif PLOT_MODE == "all_unique_pairs":
    trip_ids = sorted(route_df["trip_id"].dropna().astype(str).unique())
else:
    raise ValueError("PLOT_MODE must be first_trip, busiest_trip, or all_unique_pairs.")

stop_times = stop_times_for_trips(GTFS_ZIP, trip_ids)
trip_meta = trips.set_index("trip_id")
stop_lookup = stops.set_index("stop_id")
signal_lookup = sig_df.set_index(["from_stop_id", "to_stop_id"]).to_dict("index")

print(f"PLOT_MODE: {PLOT_MODE}")
print(f"Trip IDs used: {trip_ids[:5]}{' ...' if len(trip_ids) > 5 else ''}")

PLOT_MODE: first_trip
Trip IDs used: ['5695_40808']


## 5. Build Segment Geometry and Join Signal Counts

In [22]:
segment_rows = []
seen_pairs = set()

for trip_id in trip_ids:
    rows = stop_times.get(str(trip_id), [])
    if not rows:
        continue

    shape_points = None
    shape_id = trip_meta.loc[str(trip_id), "shape_id"] if str(trip_id) in trip_meta.index and "shape_id" in trip_meta.columns else None
    if has_shapes and pd.notna(shape_id):
        sub = shapes[shapes["shape_id"].astype(str) == str(shape_id)]
        if len(sub) >= 2:
            shape_points = shape_points_with_cumdist(sub)

    min_cum = None
    stop_positions = []
    if shape_points:
        for r in rows:
            c = stop_lookup.loc[str(r["stop_id"])]
            pos = project_stop_to_shape_m(float(c.stop_lat), float(c.stop_lon), shape_points, min_cum_m=min_cum)
            stop_positions.append(pos)
            if pos is not None:
                min_cum = pos
    else:
        stop_positions = [None] * len(rows)

    for i, (a, b) in enumerate(zip(rows[:-1], rows[1:])):
        from_id = str(a["stop_id"])
        to_id = str(b["stop_id"])
        pair_key = (from_id, to_id)
        if PLOT_MODE == "all_unique_pairs" and pair_key in seen_pairs:
            continue
        seen_pairs.add(pair_key)

        sa = stop_lookup.loc[from_id]
        sb = stop_lookup.loc[to_id]
        chord = [(float(sa.stop_lat), float(sa.stop_lon)), (float(sb.stop_lat), float(sb.stop_lon))]
        poly = None
        length_m = haversine_m(*chord[0], *chord[1])

        if shape_points:
            start_m, end_m = stop_positions[i], stop_positions[i + 1]
            if start_m is not None and end_m is not None and end_m > start_m:
                poly = shape_subpath(shape_points, start_m, end_m)
                length_m = end_m - start_m
        if not poly:
            poly = chord

        sig = signal_lookup.get(pair_key, {})
        n_signals = int(sig.get("n_signals", 0))
        source = sig.get("source", "missing")
        segment_rows.append({
            "trip_id": trip_id,
            "segment_index": len(segment_rows),
            "from_stop_id": from_id,
            "to_stop_id": to_id,
            "from_stop_name": getattr(sa, "stop_name", from_id),
            "to_stop_name": getattr(sb, "stop_name", to_id),
            "length_m": length_m,
            "n_signals": n_signals,
            "signal_source": source,
            "polyline": poly,
        })

seg = pd.DataFrame(segment_rows)
if seg.empty:
    raise ValueError("No segments were reconstructed. Check the route CSV trip IDs against the GTFS feed.")

print(f"Segments plotted: {len(seg):,}")
print(f"Total signals on plotted segments: {seg['n_signals'].sum():,}")
print(f"Segments missing from signal cache: {(seg['signal_source'] == 'missing').sum():,}")
seg[["segment_index", "from_stop_name", "to_stop_name", "length_m", "n_signals", "signal_source"]].head(10)

Segments plotted: 38
Total signals on plotted segments: 39
Segments missing from signal cache: 0


,segment_index,from_stop_name,to_stop_name,length_m,n_signals,signal_source
0,0,Ashmount Ct,Ashmount,219.618536,0,osm
1,1,Ashmount,Lotabeg Grn,268.099026,0,osm
2,2,Lotabeg Grn,Boherboy Rd,251.894790,0,osm
3,3,Boherboy Rd,Saint Joseph's Park,147.843152,0,osm
4,4,Saint Joseph's Park,Mayfield SC,332.532911,1,osm_relaxed
5,5,Mayfield SC,Glenamoy Ln,353.785847,1,osm
6,6,Glenamoy Ln,Knights Court,436.841874,2,osm
7,7,Knights Court,Mayfield Ly,390.359005,1,osm_relaxed
8,8,Mayfield Ly,St Joseph's Church,367.748669,1,osm
9,9,St Joseph's Church,Harrington Square,429.060383,1,osm


## 6. Create Interactive Map

In [23]:
all_points = [pt for poly in seg["polyline"] for pt in poly]
center = [sum(p[0] for p in all_points) / len(all_points), sum(p[1] for p in all_points) / len(all_points)]
max_sig = max(1, int(seg["n_signals"].max()))

colormap = cm.LinearColormap(
    colors=["#2E8B57", "#F2C94C", "#D64545"],
    vmin=0,
    vmax=max_sig,
    caption="Traffic signals per stop-to-stop segment",
)

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron", control_scale=True)
colormap.add_to(m)

for _, row in seg.iterrows():
    color = colormap(row["n_signals"])
    popup_html = f"""
    <b>Segment {int(row['segment_index'])}</b><br>
    {html.escape(str(row['from_stop_name']))} → {html.escape(str(row['to_stop_name']))}<br>
    <b>Traffic signals:</b> {int(row['n_signals'])}<br>
    <b>Source:</b> {html.escape(str(row['signal_source']))}<br>
    <b>Length:</b> {row['length_m']:.0f} m<br>
    <b>Stop pair:</b> {html.escape(str(row['from_stop_id']))} → {html.escape(str(row['to_stop_id']))}
    """
    folium.PolyLine(
        row["polyline"],
        color=color,
        weight=6,
        opacity=0.85,
        popup=folium.Popup(popup_html, max_width=360),
        tooltip=f"Segment {int(row['segment_index'])}: {int(row['n_signals'])} signal(s)",
    ).add_to(m)

    mid = midpoint(row["polyline"])
    label_bg = "#ffffff"
    label_border = color
    folium.Marker(
        location=mid,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                font-size: 11px;
                font-weight: 700;
                color: #1B2A4A;
                background: {label_bg};
                border: 2px solid {label_border};
                border-radius: 999px;
                width: 24px;
                height: 24px;
                line-height: 20px;
                text-align: center;
                box-shadow: 0 1px 3px rgba(0,0,0,0.25);
            ">{int(row['n_signals'])}</div>
            """
        ),
    ).add_to(m)

# Stop markers are light, so the segment colours remain the main visual layer.
stop_points = {}
for _, row in seg.iterrows():
    stop_points[row["from_stop_id"]] = row["polyline"][0]
    stop_points[row["to_stop_id"]] = row["polyline"][-1]

for stop_id, loc in stop_points.items():
    stop_name = stop_lookup.loc[stop_id].stop_name if stop_id in stop_lookup.index and "stop_name" in stop_lookup.columns else stop_id
    folium.CircleMarker(
        location=loc,
        radius=3,
        color="#1B2A4A",
        fill=True,
        fill_opacity=0.9,
        popup=f"{html.escape(str(stop_name))}<br>{html.escape(str(stop_id))}",
    ).add_to(m)

title_html = f"""
<div style="
    position: fixed;
    top: 16px;
    left: 50px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border: 1px solid #d0d7de;
    border-radius: 6px;
    font-family: Arial, sans-serif;
    font-size: 13px;
    color: #1B2A4A;
    box-shadow: 0 1px 4px rgba(0,0,0,0.18);
">
    <b>Route Traffic Signals</b><br>
    Mode: {html.escape(PLOT_MODE)}<br>
    Segments: {len(seg)} | Signals: {int(seg['n_signals'].sum())}
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

m.save(OUTPUT_HTML)
print(f"Saved map: {OUTPUT_HTML}")
m

Saved map: c:\Users\ninglin.ou\beb\notebooks\..\reports\figures\route_traffic_signals_map.html


## 7. Segment Summary Table

In [24]:
summary = (
    seg[["segment_index", "from_stop_name", "to_stop_name", "length_m", "n_signals", "signal_source"]]
    .copy()
)
summary["length_m"] = summary["length_m"].round(1)
summary.sort_values(["n_signals", "length_m"], ascending=[False, False]).head(20)

,segment_index,from_stop_name,to_stop_name,length_m,n_signals,signal_source
18,18,Daunt's Square,Washington Street,534.1,5,osm
16,16,Bridge Street,St Patrick's Street,233.9,3,osm
25,25,Victoria Cross,Dennehy's Cross,440.2,2,osm
24,24,UCC Western Gateway,Victoria Cross,439.0,2,osm
6,6,Glenamoy Ln,Knights Court,436.8,2,osm
19,19,Washington Street,Lancaster Quay,398.2,2,osm
21,21,UCC Glucksman,Western Terrace,358.6,2,osm
15,15,MacCurtain Street,Bridge Street,267.4,2,osm
17,17,St Patrick's Street,Daunt's Square,223.5,2,osm
14,14,York Hill,MacCurtain Street,194.0,2,osm
